# A3 (Pro版)：前车急刹场景 (Euro NCAP CCRb) —— 增强型 TTC 二次方程补偿与多级 AEB 闭环控制

> 🌟 **版本认证**：**IEV CloudLab V1.0 官方标准实验 (Official Standard Lab)**


> **实验定位**：本实验属于智能电动汽车（IEV）虚拟仿真体系 **链2：智能驾驶算法闭环**。
> 基于工业级 Euro NCAP CCRb (Car-to-Car Rear Braking) 急刹避撞测试标准，攻关传统线性 TTC 在前车剧烈制动工况下的“制动滞后”痛点，设计并验证**具备前车加速度辨识与二次方程解析补偿的增强型 TTC 多级紧急制动算法**。

---

### ⏱️ 3分钟极速上手指南
* **第一步 (环境自检)**：执行阶段一，加载车型参数与数学原理图，确认环境就绪；
* **第二步 (算法拆解)**：学习增强型 TTC 的二次求根公式与三级制动状态机（预充制动/部分制动/全力制动）；
* **第三步 (闭环测试)**：运行危险急刹工况仿真，观察自车在 $16\,\text{m/s}$（$57.6\,\text{km/h}$）下的安全避险过程；
* **第四步 (动态回放)**：通过 2D 鸟瞰快照、四合一遥测图表与交互式 Canvas 动效回放器，直观体察制动过程；
* **第五步 (参数对比)**：对比传统线性 TTC 与增强型 TTC 的避撞表现，验证安全巡航误触发率，并在学生动手区完成标定挑战；
* **第六步 (官方评分)**：一键执行自动评分，落盘评测结果并获取 100/100 成绩单。

---

### 📚 核心理论与工程背景：为什么前车急刹需要“增强型 TTC”？

1. **传统线性 TTC (Naive TTC) 的致命缺陷**：
   $$\text{TTC}_{\text{naive}} = \frac{d_{\text{rel}}}{v_{\text{rel}}} = \frac{x_{\text{lead}} - x_{\text{ego}} - L}{v_{\text{ego}} - v_{\text{lead}}}$$
   线性公式假设前车以**恒定速度**行驶（即假定 $a_{\text{lead}} = 0$）。
   在 CCRb 急刹工况中，两车初速度相同（$v_{\text{rel}} = 0$），前车突然施加 $-6.0\,\text{m/s}^2$ 紧急制动。
   * **在刚开始制动的瞬间**，两车速度差几乎为 0，导致 $\text{TTC}_{\text{naive}} \approx \infty$！
   * 等到两车速度差拉大、线性 TTC 终于报警并下发制动时，车间净距已经被大幅吞噬，系统极易发生严重追尾事故。

2. **增强型 TTC (Enhanced Quadratic TTC) 的运动学补偿**：
   考虑前车相对加速度 $a_{\text{rel}} = -a_{\text{lead}}$（前车减速时 $a_{\text{rel}} > 0$，代表间距加速缩短），相对位移满足运动学方程：
   $$s(t) = d_{\text{rel}} - v_{\text{rel}} \cdot t - \frac{1}{2} a_{\text{rel}} \cdot t^2 = 0$$
   整理为关于碰撞时间 $t$ 的标准一元二次方程：
   $$\frac{1}{2} a_{\text{rel}} \cdot t^2 + v_{\text{rel}} \cdot t - d_{\text{rel}} = 0$$
   当判别式 $\Delta = v_{\text{rel}}^2 + 2 a_{\text{rel}} d_{\text{rel}} > 0$ 且 $a_{\text{rel}} > 0.1$ 时，解析求解正实根即为真实碰撞时间：
   $$t_{\text{enhanced}} = \frac{-v_{\text{rel}} + \sqrt{v_{\text{rel}}^2 + 2 a_{\text{rel}} d_{\text{rel}}}}{a_{\text{rel}}}$$

3. **三级制动与防抖滞环 (Staged Braking with Hysteresis)**：
   * **Stage 1 (预充与警示)**：$\text{TTC} < 4.0\,\text{s} \implies a_{\text{cmd}} = -4.0\,\text{m/s}^2$
   * **Stage 2 (部分制动)**：$\text{TTC} < 2.5\,\text{s} \implies a_{\text{cmd}} = -6.0\,\text{m/s}^2$
   * **Stage 3 (全力制动)**：$\text{TTC} < 1.8\,\text{s} \implies a_{\text{cmd}} = -8.0\,\text{m/s}^2$
   * **滞环释放裕度 (Hysteresis Margin)**：$\Delta t_{\text{hyst}} = 0.3\,\text{s}$，仅当 TTC 恢复至阈值 $+ 0.3\,\text{s}$ 时才降级释放，防止制动器抖动磨损。

> 注：鸟瞰图中车道横向位置为示意，车辆纵向位置与时刻均取自真实仿真轨迹。


---

## 阶段零：新手直觉探索与课前导学 (Phase 0: Interactive Intuition Explorer)

> 💡 **给初学者的直观模式 (Novice Mode)**：
> - **零代码调参**：拖动下方滑块调节前车减速度与 AEB 触发 TTC 阈值，点击 **“🚀 对比制动避障动态”**；
> - **三件套即时反馈**：同图查看朴素 TTC（匀速假设）vs 增强 TTC（前车减速度补偿）在制动距离与碰撞时速上的显著差异；
> - **寻优微任务**：在前车急刹（如 -4.0 m/s² ~ -5.0 m/s²）工况下，观察朴素 TTC 为何导致严重追尾，找出增强 TTC 能实现零碰撞刹停的安全阈值区间，输入心得并点击 **“📝 预习打卡保存”**。
> - *(注：本环节为预习导学打卡，完成即保存预习记录（是否计入平时成绩由教师定，当前系统不自动计分），不影响后续阶段六的正式 29+1 实验报告成绩)*
> - **口径说明：本环节为纯 Python 运动学 TTC 演示（非 esmini 闭环），真引擎闭环在后续阶段。


In [ ]:
# 运行直观交互探索驾驶舱
import sys, os
for p in ["/home/jovyan/labs", os.path.abspath("../labs"), os.path.abspath("labs")]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
        break

from common.lab_explorers import show_a3_explorer
show_a3_explorer()

In [1]:
# 阶段一：环境自检与数学原理图展示
import sys, os
sys.dont_write_bytecode = True

# 1. 稳健的多路径导入保障
for p in ['/home/jovyan/labs', os.path.abspath('../labs'), os.path.abspath('labs')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
        break

import matplotlib.pyplot as plt
from common.controller import BaseController
from common.esmini_env import EsminiEnv
from common.vehicle_library import load_vehicle
from common.diagrams import ttc_principle, aeb_block_diagram

# 2. 从标准车辆库载入参数（链1：车型设计 -> 链2：智能驾驶）
veh_params = load_vehicle('bev_sedan')
l_wb = veh_params['kinematic']['l_f'] + veh_params['kinematic']['l_r']
print(f"✓ 实验车型加载成功: {veh_params['name']} (整备质量: {veh_params['mass_kg']} kg)")
print(f"  动力学边界: 最大减速度 = {veh_params['kinematic']['max_accel_mss']} m/s², 轴距 = {l_wb:.2f} m")

# 3. 展示 TTC 碰撞时间数学原理图与系统控制流图
fig_ttc = ttc_principle()
plt.show()

fig_aeb = aeb_block_diagram()
plt.show()
print("✓ 平台环境与原理示意图就绪")


✓ 实验车型加载成功: 标准纯电轿车 (整备质量: 1800 kg)
  动力学边界: 最大减速度 = 3.0 m/s², 轴距 = 2.80 m
✓ 平台环境与原理示意图就绪


## 阶段二：增强型 AEB 控制器设计 | Controller Implementation

控制器核心设计规范：
1. **前车加速度低通估计**：采用一阶滤波 $\hat{a}_{\text{lead}}^{(k)} = 0.7 \hat{a}_{\text{lead}}^{(k-1)} + 0.3 a_{\text{raw}}$，有效滤除雷达差分噪声；
2. **二次判别式求解**：对前车减速阶段精准计算二次根，前车匀速或加速时平滑降级至线性 TTC；
3. **分级状态保持**：引入滞环释放逻辑，并记录激活次数以准确度量误触发率。


In [2]:
class FullAEBController(BaseController):
    """具备前车加速度补偿的增强型三级 AEB 控制器"""

    def __init__(self, config=None):
        super().__init__(config)
        # 三级 TTC 触发阈值（秒）
        self.t_warn = self.config.get('t_warn', 4.0)    # 一级：预警与轻度减速
        self.t_part = self.config.get('t_part', 2.5)    # 二级：部分减速
        self.t_full = self.config.get('t_full', 1.8)    # 三级：全力紧急制动
        # 滞环释放裕度
        self.hyst = self.config.get('hysteresis', 0.3)
        # 计算模式：'enhanced' (二次补偿) 或 'naive' (线性基准)
        self.mode = self.config.get('mode', 'enhanced')
        
        self._active_level = 0
        self.activations = 0
        self._prev_lead_v = None
        self._lead_a_est = 0.0

    def reset(self):
        self._active_level = 0
        self.activations = 0
        self._prev_lead_v = None
        self._lead_a_est = 0.0

    def _compute_ttc(self, ego, lead, dt):
        """计算 TTC 与车间净间距"""
        rel_dist = abs(lead['x'] - ego['x']) - 4.5  # 扣除车长，求前后保险杠净距
        rel_v = ego['v'] - lead['v']
        
        # 滤波估计前车加速度
        if self._prev_lead_v is not None:
            lead_a_raw = (lead['v'] - self._prev_lead_v) / dt
            self._lead_a_est = 0.7 * self._lead_a_est + 0.3 * lead_a_raw
        self._prev_lead_v = lead['v']

        # 线性基线模式
        if self.mode == 'naive':
            ttc_naive = rel_dist / rel_v if rel_v > 0.1 else float('inf')
            return ttc_naive, rel_dist

        # 增强模式：求解二次方程 0.5 * rel_a * t^2 + rel_v * t - rel_dist = 0
        rel_a = -self._lead_a_est  # 前车减速时 rel_a 为正，间距加速收缩
        
        if rel_v <= 0.1 and rel_a <= 0.1:
            return float('inf'), rel_dist
            
        if rel_a > 0.1:
            disc = rel_v ** 2 + 2 * rel_a * rel_dist
            if disc <= 0:
                return float('inf'), rel_dist
            ttc_enhanced = (-rel_v + disc ** 0.5) / rel_a
            return max(0.0, ttc_enhanced), rel_dist
            
        return (rel_dist / rel_v if rel_v > 0.1 else float('inf')), rel_dist

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        dt = observation.get('dt', 0.05)
        
        if not targets:
            return {'target_a': 0.0, 'steer_angle': 0.0, 'level': 0, 'ttc': float('inf'), 'lead_a': 0.0}

        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        ttc, rel_dist = self._compute_ttc(ego, lead, dt)

        # 具备滞环逻辑的三级制动判定
        if ttc < self.t_full:
            level = 3
        elif ttc < self.t_part:
            level = 2
        elif ttc < self.t_warn:
            level = 1
        else:
            if self._active_level == 3 and ttc < self.t_full + self.hyst:
                level = 3
            elif self._active_level == 2 and ttc < self.t_part + self.hyst:
                level = 2
            elif self._active_level == 1 and ttc < self.t_warn + self.hyst:
                level = 1
            else:
                level = 0

        # 记录新激活次数（用于误触发统计）
        if level > 0 and self._active_level == 0:
            self.activations += 1
        self._active_level = level

        # 映射制动减速度
        accel_map = {0: 0.0, 1: -4.0, 2: -6.0, 3: -8.0}
        cmd_a = accel_map[level]
        
        return {
            'target_a': cmd_a,
            'steer_angle': 0.0,
            'level': level,
            'ttc': ttc,
            'rel_dist': rel_dist,
            'lead_a': self._lead_a_est
        }

print("✓ FullAEBController 控制器类构建就绪")


✓ FullAEBController 控制器类构建就绪


## 阶段三：运行闭环仿真 | Run Closed-Loop Simulation

本阶段构建模块化仿真执行器 `run_aeb_simulation`：
* 支持测试两类经典工况：
  1. **危险急刹场景 (`danger`)**：基于 `aeb_brake.xosc`，前车由 $16\,\text{m/s}$ 紧急深刹停住，检验避撞性能与最终安全净距；
  2. **安全巡航场景 (`safe`)**：基于 `aeb_cruise.xosc`，自车 $12\,\text{m/s}$，前车 $14\,\text{m/s}$，间距持续拉大，检验 AEB 是否保持静默（误触发率 $\le 5\%$）。


In [3]:
def run_aeb_simulation(scenario_type='danger', mode='enhanced', t_warn=4.0, t_part=2.5, t_full=1.8, steps=400):
    """运行单次 AEB 闭环仿真，返回完整遥测轨迹与关键指标"""
    # 自动探测场景文件路径
    xosc_name = 'aeb_brake.xosc' if scenario_type == 'danger' else 'aeb_cruise.xosc'
    scenario_candidates = [
        f'/home/jovyan/labs/scenarios/A_esmini/A3_aeb/{xosc_name}',
        f'labs/scenarios/A_esmini/A3_aeb/{xosc_name}',
        f'../labs/scenarios/A_esmini/A3_aeb/{xosc_name}'
    ]
    scenario_file = next((p for p in scenario_candidates if os.path.exists(p)), scenario_candidates[0])
    road_file = scenario_file.replace('.xosc', '.xodr') if os.path.exists(scenario_file.replace('.xosc', '.xodr')) else None

    controller = FullAEBController(config={
        'mode': mode,
        't_warn': t_warn,
        't_part': t_part,
        't_full': t_full,
        'hysteresis': 0.3
    })

    env = EsminiEnv(scenario=scenario_file, road=road_file, dt=0.05)
    ego_v_init = 16.0 if scenario_type == 'danger' else 12.0
    obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=ego_v_init)
    controller.reset()

    traj = []
    min_ttc, min_gap, is_collision = float('inf'), float('inf'), False
    brake_active_steps = 0

    for step in range(steps):
        action = controller.step(obs)
        obs, done, info = env.step(action)
        t = obs['sim_time']
        ego = obs['ego']
        
        tgt_x, tgt_v, tgt_a = None, None, 0.0
        gap = float('inf')
        for tgt in obs.get('targets', []):
            rel_d = abs(tgt['x'] - ego['x']) - 4.5
            gap = min(gap, rel_d)
            min_gap = min(min_gap, rel_d)
            tgt_x, tgt_v = tgt['x'], tgt['v']
            if rel_d < 0.5 and not is_collision:
                is_collision = True

        ttc_val = action.get('ttc', float('inf'))
        if ttc_val < min_ttc and ttc_val > 0.01:
            min_ttc = ttc_val

        if abs(action['target_a']) > 0.5:
            brake_active_steps += 1

        traj.append({
            't': t,
            'ego_x': ego['x'],
            'ego_v': ego['v'],
            'tgt_x': tgt_x,
            'tgt_v': tgt_v,
            'gap': gap,
            'ttc': ttc_val,
            'target_a': action['target_a'],
            'stage': action.get('level', 0)
        })

        if done or is_collision:
            break

    env.close()
    false_trigger_rate = brake_active_steps / max(1, len(traj))

    return {
        'scenario_type': scenario_type,
        'mode': mode,
        'min_ttc': min_ttc,
        'min_gap': min_gap,
        'is_collision': is_collision,
        'false_trigger_rate': false_trigger_rate,
        'activations': controller.activations,
        'trajectory': traj
    }

# 运行基准危险急刹实验（增强型 TTC）
baseline_danger = run_aeb_simulation(scenario_type='danger', mode='enhanced')

print("=" * 60)
print(f"✓ 危险急刹闭环仿真完成！步数: {len(baseline_danger['trajectory'])} 步 (时长: {baseline_danger['trajectory'][-1]['t']:.2f}s)")
print(f"  • 最小计算 TTC: {baseline_danger['min_ttc']:.2f} s")
print(f"  • 最终静止/最小净间距: {baseline_danger['min_gap']:.2f} m (安全充裕)")
print(f"  • 碰撞状态: {'💥 发生碰撞!' if baseline_danger['is_collision'] else '🛡️ 成功避撞刹停 (无碰撞)'}")
print("=" * 60)


esmini GIT REV: v3.10.0-0-06065f92
esmini GIT TAG: v3.10.0
esmini GIT BRANCH: tags/v3.10.0^0
esmini BUILD VERSION: 6442
[] [info] Ignoring missing default config: /opt/esmini/esmini/bin/../config.yml
[] [info] Loaded aeb_brake.xosc
Unable to open log file log.txt
Unable to open log file log.txt
[2026-10-27 14:06:28]
Unable to open log file log.txt
[] [info] Player options: --osc aeb_brake.xosc --camera_mode orbit --fixed_timestep 0.05 --ground_plane auto --headless --info_text 1 --logfile_path log.txt --log_level info --text_scale 1.0 --traj_filter 0.1 --light_mode auto 
Unable to open log file log.txt
[] [info] Run simulation decoupled from realtime, with fixed timestep: 0.05
Unable to open log file log.txt
[] [info] Generated seed 3487807174
Unable to open log file log.txt
[] [info] Loading aeb_brake.xosc (v1.1)
Unable to open log file log.txt
[] [info] 0 variables
Unable to open log file log.txt
[] [info] Loaded OpenDRIVE: aeb.xodr
Unable to open log file log.txt
[] [info] Non zero 

## 阶段四：直观多角度场景可视化 | Comprehensive Visualization

本阶段提供三重维度场景感知：
1. **2D 鸟瞰路面快照 (BEV Multi-Stage Snapshot)**：直观展现初始巡航、前车急刹减速、自车安全刹停三大关键时序帧；
2. **四合一多轴遥测仪表板**：涵盖两车速度、车间净距、三级制动阶梯、TTC 动态演化；
3. **轻量级原生 HTML5/Canvas 交互式 2D 动态回放器**：带时间轴滑块、多倍速播放、尾灯高亮与实时 HUD 遥测仪表。


In [4]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import HTML, display

def plot_aeb_dashboard(res):
    """绘制四合一工程遥测图表与路面鸟瞰图"""
    traj = res['trajectory']
    times = [d['t'] for d in traj]
    ego_v = [d['ego_v'] for d in traj]
    tgt_v = [d['tgt_v'] if d['tgt_v'] is not None else 0.0 for d in traj]
    gaps = [d['gap'] for d in traj]
    accels = [d['target_a'] for d in traj]
    ttcs = [min(d['ttc'], 10.0) for d in traj]

    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.3])

    # 1. 2D 鸟瞰路面多阶段快照
    ax_bev = fig.add_subplot(gs[0, :])
    ax_bev.set_facecolor('#1e293b')
    ax_bev.axhline(0.0, color='white', lw=2)
    ax_bev.axhline(3.75, color='#fbbf24', linestyle='--', lw=1.5, alpha=0.8)
    ax_bev.axhline(7.5, color='white', lw=2)

    snapshot_times = [0.0, 3.0, times[-1]]
    stage_titles = ['t=0.0s 初始巡航 (57.6km/h)', 't=3.0s 前车急刹介入', f't={times[-1]:.1f}s 安全避险刹停']
    ego_colors = ['#60a5fa', '#2563eb', '#1d4ed8']
    tgt_colors = ['#f87171', '#dc2626', '#991b1b']

    for idx, target_t in enumerate(snapshot_times):
        frame = min(traj, key=lambda d: abs(d['t'] - target_t))
        ex = frame['ego_x']
        tx = frame['tgt_x'] if frame['tgt_x'] is not None else ex + 20.0
        
        # 绘制 Ego 车辆
        ax_bev.add_patch(patches.Rectangle((ex, 2.75), 4.8, 2.0, color=ego_colors[idx], ec='white', lw=1.2))
        ax_bev.text(ex + 2.4, 3.75, "Ego", ha='center', va='center', color='white', fontsize=8, fontweight='bold')
        
        # 绘制 Target 车辆
        ax_bev.add_patch(patches.Rectangle((tx, 2.75), 4.8, 2.0, color=tgt_colors[idx], ec='white', lw=1.2))
        ax_bev.text(tx + 2.4, 3.75, "Target", ha='center', va='center', color='white', fontsize=8, fontweight='bold')

        # 标注阶段指示与间距
        cur_gap = tx - ex - 4.5
        ax_bev.text(ex, 6.2, f"{stage_titles[idx]} | 净距: {cur_gap:.1f}m", color='#f8fafc', fontsize=8.5,
                    bbox=dict(boxstyle="round,pad=0.25", fc="#334155", ec="#64748b", alpha=0.9))

    ax_bev.set_xlim(0, max(traj[-1]['tgt_x'] + 15, 120))
    ax_bev.set_ylim(0.5, 7.5)
    ax_bev.set_yticks([3.75])
    ax_bev.set_yticklabels(['主车道 (Ego Lane)'], fontsize=9.5)
    ax_bev.set_xlabel('纵向位置 X (m)', fontsize=9.5)
    ax_bev.set_title("Euro NCAP CCRb 急刹工况 2D 鸟瞰演化快照 (BEV Multi-Stage Snapshot)", fontsize=11, fontweight='bold')
    ax_bev.grid(True, axis='x', alpha=0.2, linestyle=':')

    # 2. 下半部分左图：速度响应曲线
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.plot(times, ego_v, color='#2563eb', lw=2.2, label='Ego Speed (主车速度)')
    ax1.plot(times, tgt_v, color='#dc2626', lw=2.0, linestyle='--', label='Target Speed (前车速度)')
    ax1.axhline(0, color='gray', linestyle=':', alpha=0.7)
    ax1.set_xlabel('时间 Time (s)', fontsize=9.5)
    ax1.set_ylabel('速度 Speed (m/s)', fontsize=9.5)
    ax1.set_title('速度变化与制动响应曲线', fontsize=10.5, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8.5)

    # 3. 下半部分右图：三级减速度与间距曲线
    ax2 = fig.add_subplot(gs[1, 1])
    ax2_twin = ax2.twinx()
    
    line1 = ax2.plot(times, accels, color='#ea580c', lw=2.0, label='Command Accel (指令减速度)')
    ax2.axhline(-4.0, color='#f59e0b', linestyle=':', alpha=0.7, label='Stage 1 (-4 m/s²)')
    ax2.axhline(-6.0, color='#ea580c', linestyle=':', alpha=0.7, label='Stage 2 (-6 m/s²)')
    ax2.axhline(-8.0, color='#dc2626', linestyle=':', alpha=0.7, label='Stage 3 (-8 m/s²)')
    
    line2 = ax2_twin.plot(times, gaps, color='#059669', lw=2.0, linestyle='-.', label='Net Gap (前后保险杠净距)')
    ax2_twin.set_ylabel('净距 Net Gap (m)', color='#059669', fontsize=9.5)

    ax2.set_xlabel('时间 Time (s)', fontsize=9.5)
    ax2.set_ylabel('加速度 (m/s²)', color='#ea580c', fontsize=9.5)
    ax2.set_title('三级制动阶梯与动态净距演变', fontsize=10.5, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 合并双坐标轴图例
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='lower left', fontsize=8)

    plt.tight_layout()
    plt.show()

def render_interactive_player(res):
    """生成并渲染轻量级原生 HTML5/Canvas 交互式 2D 动态回放器"""
    traj = res['trajectory']
    step = max(1, len(traj) // 100)
    sampled = traj[::step]
    if traj and sampled[-1] != traj[-1]:
        sampled.append(traj[-1])

    data_json = json.dumps([{
        't': round(d['t'], 2),
        'ego_x': round(d['ego_x'], 1),
        'ego_v': round(d['ego_v'], 1),
        'tgt_x': round(d['tgt_x'] if d['tgt_x'] is not None else 0.0, 1),
        'tgt_v': round(d['tgt_v'] if d['tgt_v'] is not None else 0.0, 1),
        'gap': round(d['gap'], 1),
        'ttc': round(min(d['ttc'], 99.9), 1),
        'a': round(d['target_a'], 1),
        'stage': d['stage']
    } for d in sampled])

    html_code = f"""
    <div id="a3_player_wrap" style="background: #0f172a; border-radius: 10px; padding: 14px; font-family: system-ui, sans-serif; color: #f8fafc; max-width: 820px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.4); margin-top: 15px;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
        <div style="font-weight: bold; font-size: 14px; display: flex; align-items: center; gap: 8px;">
          <span style="color: #38bdf8;">🚗 A3 CCRb 动态交互回放器</span>
          <span id="a3_status" style="background: #334155; padding: 2px 8px; border-radius: 4px; font-size: 11px;">准备就绪</span>
        </div>
        <div style="font-size: 12px; color: #94a3b8; font-family: monospace;">
          t: <span id="a3_t" style="color: #f8fafc; font-weight: bold;">0.00</span>s | 
          主车: <span id="a3_v" style="color: #38bdf8; font-weight: bold;">0.0</span> m/s | 
          前车: <span id="a3_tv" style="color: #f87171; font-weight: bold;">0.0</span> m/s | 
          净距: <span id="a3_gap" style="color: #fbbf24; font-weight: bold;">0.0</span> m | 
          TTC: <span id="a3_ttc" style="color: #4ade80; font-weight: bold;">0.0</span> s
        </div>
      </div>
      <canvas id="a3_canvas" width="790" height="110" style="width: 100%; height: 110px; background: #1e293b; border-radius: 6px; display: block;"></canvas>
      <div style="display: flex; align-items: center; gap: 10px; margin-top: 10px;">
        <button id="a3_btn_play" style="background: #2563eb; color: white; border: none; padding: 5px 14px; border-radius: 5px; cursor: pointer; font-size: 12px; font-weight: bold;">▶ 播放</button>
        <button id="a3_btn_reset" style="background: #475569; color: white; border: none; padding: 5px 12px; border-radius: 5px; cursor: pointer; font-size: 12px;">⏪ 复位</button>
        <input id="a3_slider" type="range" min="0" max="{len(sampled)-1}" value="0" style="flex: 1; accent-color: #38bdf8; cursor: pointer;">
        <select id="a3_speed" style="background: #334155; color: #f8fafc; border: 1px solid #475569; padding: 3px 6px; border-radius: 4px; font-size: 11px;">
          <option value="0.5">0.5x</option>
          <option value="1" selected>1.0x</option>
          <option value="2">2.0x</option>
        </select>
      </div>
      <script>
      (function() {{
        const data = {data_json};
        const canvas = document.getElementById('a3_canvas');
        if (!canvas) return;
        const ctx = canvas.getContext('2d');
        const slider = document.getElementById('a3_slider');
        const btnPlay = document.getElementById('a3_btn_play');
        const btnReset = document.getElementById('a3_btn_reset');
        const speedSel = document.getElementById('a3_speed');
        const txtStatus = document.getElementById('a3_status');
        const txtT = document.getElementById('a3_t');
        const txtV = document.getElementById('a3_v');
        const txtTV = document.getElementById('a3_tv');
        const txtGap = document.getElementById('a3_gap');
        const txtTTC = document.getElementById('a3_ttc');
        let curIdx = 0;
        let isPlaying = false;
        let timer = null;

        function renderFrame(idx) {{
          curIdx = idx;
          slider.value = idx;
          const d = data[idx];
          txtT.textContent = d.t.toFixed(2);
          txtV.textContent = d.ego_v.toFixed(1);
          txtTV.textContent = d.tgt_v.toFixed(1);
          txtGap.textContent = d.gap.toFixed(1);
          txtTTC.textContent = d.ttc >= 50 ? '∞' : d.ttc.toFixed(1);

          if (d.ego_v <= 0.1 && d.t > 4) {{
            txtStatus.textContent = '✅ 安全刹停避险';
            txtStatus.style.background = '#059669';
          }} else if (d.stage === 3) {{
            txtStatus.textContent = '🚨 三级全力紧急制动 (-8m/s²)';
            txtStatus.style.background = '#dc2626';
          }} else if (d.stage === 2) {{
            txtStatus.textContent = '⚠️ 二级部分制动 (-6m/s²)';
            txtStatus.style.background = '#ea580c';
          }} else if (d.stage === 1) {{
            txtStatus.textContent = '⚡ 一级预充制动 (-4m/s²)';
            txtStatus.style.background = '#d97706';
          }} else {{
            txtStatus.textContent = '🚗 匀速巡航跟随中';
            txtStatus.style.background = '#334155';
          }}

          const W = canvas.width, H = canvas.height;
          ctx.clearRect(0, 0, W, H);
          
          // 道路路面绘制
          ctx.fillStyle = '#0f172a';
          ctx.fillRect(0, 20, W, 70);
          ctx.strokeStyle = '#ffffff';
          ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.moveTo(0, 20); ctx.lineTo(W, 20);
          ctx.moveTo(0, 90); ctx.lineTo(W, 90);
          ctx.stroke();

          // 车道分界线
          ctx.strokeStyle = '#64748b';
          ctx.setLineDash([12, 10]);
          ctx.lineWidth = 1;
          ctx.beginPath();
          ctx.moveTo(0, 55); ctx.lineTo(W, 55);
          ctx.stroke();
          ctx.setLineDash([]);

          // 坐标变换：以两车中点为视角中心
          const camX = (d.ego_x + d.tgt_x) / 2;
          const scale = 4.2;
          const offsetX = W / 2 - camX * scale;
          function toScreenX(x) {{ return x * scale + offsetX; }}

          const egoScreenX = toScreenX(d.ego_x);
          const tgtScreenX = toScreenX(d.tgt_x);
          const laneY = 55;

          // 绘制 Ego 车辆
          ctx.fillStyle = '#2563eb';
          ctx.strokeStyle = '#93c5fd';
          ctx.lineWidth = 1.5;
          ctx.fillRect(egoScreenX - 22, laneY - 10, 44, 20);
          ctx.strokeRect(egoScreenX - 22, laneY - 10, 44, 20);

          // Ego 刹车高亮尾灯
          if (d.a < -1.0) {{
            ctx.fillStyle = '#ef4444';
            ctx.shadowColor = '#ef4444';
            ctx.shadowBlur = 8;
            ctx.fillRect(egoScreenX - 24, laneY - 8, 3, 5);
            ctx.fillRect(egoScreenX - 24, laneY + 3, 3, 5);
            ctx.shadowBlur = 0;
          }}

          ctx.fillStyle = '#ffffff';
          ctx.font = 'bold 10px sans-serif';
          ctx.fillText('Ego', egoScreenX - 10, laneY + 3.5);

          // 绘制 Target 车辆
          ctx.fillStyle = '#dc2626';
          ctx.strokeStyle = '#fca5a5';
          ctx.lineWidth = 1.5;
          ctx.fillRect(tgtScreenX - 22, laneY - 10, 44, 20);
          ctx.strokeRect(tgtScreenX - 22, laneY - 10, 44, 20);

          // 前车急刹高亮尾灯
          if (d.t >= 2.0 && d.tgt_v < 15.5 && d.tgt_v > 0.05) {{
            ctx.fillStyle = '#ef4444';
            ctx.shadowColor = '#ef4444';
            ctx.shadowBlur = 10;
            ctx.fillRect(tgtScreenX - 24, laneY - 8, 3, 5);
            ctx.fillRect(tgtScreenX - 24, laneY + 3, 3, 5);
            ctx.shadowBlur = 0;
          }}

          ctx.fillStyle = '#ffffff';
          ctx.font = 'bold 10px sans-serif';
          ctx.fillText('Lead', tgtScreenX - 12, laneY + 3.5);

          // 连线与距离标尺
          if (tgtScreenX > egoScreenX) {{
            ctx.strokeStyle = '#fbbf24';
            ctx.lineWidth = 1.5;
            ctx.beginPath();
            ctx.moveTo(egoScreenX + 22, laneY);
            ctx.lineTo(tgtScreenX - 22, laneY);
            ctx.stroke();
            ctx.fillStyle = '#fbbf24';
            ctx.font = 'bold 11px sans-serif';
            ctx.fillText(d.gap.toFixed(1) + 'm', (egoScreenX + tgtScreenX) / 2 - 14, laneY - 6);
          }}
        }}

        slider.addEventListener('input', (e) => {{
          renderFrame(parseInt(e.target.value));
        }});

        function playNext() {{
          if (!isPlaying) return;
          if (curIdx < data.length - 1) {{
            renderFrame(curIdx + 1);
            const speed = parseFloat(speedSel.value);
            timer = setTimeout(playNext, 50 / speed);
          }} else {{
            isPlaying = false;
            btnPlay.textContent = '▶ 播放';
          }}
        }}

        btnPlay.addEventListener('click', () => {{
          if (isPlaying) {{
            isPlaying = false;
            btnPlay.textContent = '▶ 播放';
            clearTimeout(timer);
          }} else {{
            if (curIdx >= data.length - 1) curIdx = 0;
            isPlaying = true;
            btnPlay.textContent = '⏸ 暂停';
            playNext();
          }}
        }});

        btnReset.addEventListener('click', () => {{
          isPlaying = false;
          btnPlay.textContent = '▶ 播放';
          clearTimeout(timer);
          renderFrame(0);
        }});

        renderFrame(0);
      }})();
      </script>
    </div>
    """
    display(HTML(html_code))

plot_aeb_dashboard(baseline_danger)
render_interactive_player(baseline_danger)


## 阶段五：参数调优实验室与对比实验 | Parameter Lab & Comparisons

### 🔬 实验 5.1：增强型 TTC vs 传统线性 TTC 极端急刹硬碰硬
在相同急刹场景下，对比开启前车加速度二次补偿（Enhanced）与关闭补偿（Naive）的制动时机与避撞结果。

### 🛡️ 实验 5.2：安全场景巡航误触发率检验 (False-Trigger Test)
在安全工况下（自车 $12\,\text{m/s}$ 跟随 $14\,\text{m/s}$ 前车，车距持续拉大），验证 AEB 不会发生不必要的幽灵刹车（平台验收指标：误触发率 $\le 5\%$）。

### 🎯 实验 5.3：学生动手标定挑战 (Hands-on Calibration Challenge)
自主调节分级阈值，寻找制动平顺性与安全裕度之间的最优平衡点。


In [5]:
# ----------------------------------------------------
# 1. 增强型 TTC vs 传统线性 TTC 对比实验
# ----------------------------------------------------
print("=" * 68)
print("正在执行 对比实验：增强型 TTC (二次补偿) vs 线性 TTC (无加速度补偿)...")
naive_danger = run_aeb_simulation(scenario_type='danger', mode='naive')

print("=" * 68)
print(f"{'对比维度':<16} | {'增强型 TTC (Enhanced)':<18} | {'传统线性 TTC (Naive)':<18}")
print("-" * 68)
print(f"{'是否发生碰撞':<16} | {'🛡️ 安全无碰撞' if not baseline_danger['is_collision'] else '💥 碰撞':<18} | {'🛡️ 无碰撞' if not naive_danger['is_collision'] else '💥 追尾碰撞!':<18}")
print(f"{'最小剩余净间距':<16} | {baseline_danger['min_gap']:<18.2f}m | {naive_danger['min_gap']:<18.2f}m")
print(f"{'最小计算 TTC':<16} | {baseline_danger['min_ttc']:<18.2f}s | {naive_danger['min_ttc']:<18.2f}s")
print("=" * 68)

# 叠画速度响应对比曲线
plt.figure(figsize=(10, 4.2))
plt.plot([d['t'] for d in baseline_danger['trajectory']], [d['ego_v'] for d in baseline_danger['trajectory']], 
         color='#2563eb', lw=2.2, label=f"Enhanced TTC (Safe Stop, Min Gap {baseline_danger['min_gap']:.1f}m)")
plt.plot([d['t'] for d in naive_danger['trajectory']], [d['ego_v'] for d in naive_danger['trajectory']], 
         color='#dc2626', lw=2.0, linestyle='--', label=f"Naive TTC (Delayed Brake, Min Gap {naive_danger['min_gap']:.1f}m)")
plt.plot([d['t'] for d in baseline_danger['trajectory']], [d['tgt_v'] for d in baseline_danger['trajectory']], 
         color='gray', lw=1.5, linestyle=':', label="Lead Vehicle (Hard Brake)")
plt.axhline(0, color='black', lw=0.8, linestyle='--')
plt.xlabel('Time (s)', fontsize=9.5)
plt.ylabel('Speed (m/s)', fontsize=9.5)
plt.title('Braking Response Comparison: Enhanced Quadratic TTC vs Naive Linear TTC', fontsize=11, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

# ----------------------------------------------------
# 2. 安全巡航工况误触发率检验
# ----------------------------------------------------
print("\n正在执行 安全巡航场景误触发率检验 (aeb_cruise.xosc)...")
safe_res = run_aeb_simulation(scenario_type='safe', mode='enhanced', steps=200)

print(f"✓ 安全巡航测试完成: 运行步数 = {len(safe_res['trajectory'])} 步")
print(f"  • 误触发率 (False-Trigger Rate): {safe_res['false_trigger_rate']*100:.2f}% (考核标准 < 5.0%)")
print(f"  • AEB 激活次数: {safe_res['activations']} 次 (必须为 0 次)")
assert safe_res['false_trigger_rate'] < 0.05, "安全巡航场景出现误触发制动！"
print("✓ 安全巡航验收通过，零误触发！")

# ----------------------------------------------------
# 3. 🎯 学生动手标定挑战 (Hands-on Calibration Challenge)
# ----------------------------------------------------
print("\n💡 [学生动手标定挑战] 寻找舒适度与安全边界的最佳折中：")
print("尝试微调下面的阈值参数，观察在 Euro NCAP 急刹下，能否保持最小净距 >= 5.0m 同时避免过早惊吓驾驶员：")

my_t_warn = 3.8     # 尝试调节：3.2 ~ 4.2 s
my_t_part = 2.4     # 尝试调节：2.0 ~ 2.8 s
my_t_full = 1.6     # 尝试调节：1.4 ~ 2.0 s

custom_res = run_aeb_simulation(scenario_type='danger', mode='enhanced', 
                                t_warn=my_t_warn, t_part=my_t_part, t_full=my_t_full)
print(f"👉 自定义标定结果: 最小净间隙 = {custom_res['min_gap']:.2f} m, 最小 TTC = {custom_res['min_ttc']:.2f} s, 碰撞 = {custom_res['is_collision']}")


正在执行 对比实验：增强型 TTC (二次补偿) vs 线性 TTC (无加速度补偿)...
[0.050] [info] act_start: true, delay: 0.00, 0.0500 > 0.0000, edge: rising
[0.050] [info] Trigger /------------------------------------------------
[0.050] [info] Group 0:
[0.050] [info] act_start: true
[0.050] [info] Trigger  ------------------------------------------------/
[0.050] [info] brake_act standbyState -> startTransition -> runningState
[0.050] [info] brake_group initState -> startTransition -> runningState
[0.050] [info] brake_maneuver initState -> startTransition -> runningState
[0.050] [info] brake_event initState -> initToStandbyTransition -> standbyState
[3.050] [info] brake_trigger: true, delay: 0.00, 3.0500 > 3.0000, edge: rising
[3.050] [info] Trigger /------------------------------------------------
[3.050] [info] Group 0:
[3.050] [info] brake_trigger: true
[3.050] [info] Trigger  ------------------------------------------------/
[3.050] [info] brake_event standbyState -> startTransition -> runningState
[3.050] [info] h

## 阶段六：官方评分与结果归档 | Auto-Grading & Results

本阶段调用 IEV CloudLab 官方评测套件：
* **考核维度 1**：增强型 TTC 急刹工况安全无碰撞 (`collision == False`, 权重 3)；
* **考核维度 2**：安全巡航工况误触发率达标 (`false_trigger_rate < 0.05`, 权重 1)；
* 评测结果自动保存为 JSON 文件，无缝对接平台成绩汇总系统与教师控制台。


In [6]:
import json
from common.evaluator import grade_metrics

# 1. 整理基线达标指标并落盘
save_dir = '/home/jovyan/labs/data/results'
record = {
    'experiment': 'A3_aeb_closedloop',
    'metrics': {
        'collision': baseline_danger['is_collision'],
        'false_trigger_rate': round(safe_res['false_trigger_rate'], 4)
    }
}

saved_ok = False
for target_dir in [save_dir, 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'A3_aeb_closedloop.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f"✓ 结果文件已成功归档至: {out_file}")
        saved_ok = True
        break
    except OSError:
        continue

if not saved_ok:
    print("ℹ️ 提示: 当前运行环境为只读挂载模式，跳过落盘，直接进入评测")

# 2. 调用平台官方评测器输出成绩单
eval_report = grade_metrics('A3_aeb_closedloop', record['metrics'])

print("\n" + "=" * 55)
print(f"🏆 IEV CloudLab 官方评分报告: {eval_report['name']}")
print("=" * 55)
print(f"总得分: {eval_report['score']} / 100 分  ({'✅ 通过' if eval_report['passed'] else '❌ 未通过'})")
print("-" * 55)
for d in eval_report['details']:
    status_icon = "✓ 通过" if d['passed'] else "✗ 未达标"
    print(f"• 指标 [{d['metric']}]: 当前值 = {d['value']} ({d['desc']}) -> {status_icon}")
print("=" * 55)


✓ 结果文件已成功归档至: labs/data/results/A3_aeb_closedloop.json

🏆 IEV CloudLab 官方评分报告: AEB增强TTC闭环
总得分: 100.0 / 100 分  (✅ 通过)
-------------------------------------------------------
• 指标 [collision]: 当前值 = False (增强TTC版无碰撞) -> ✓ 通过
• 指标 [false_trigger_rate]: 当前值 = 0.0 (误触发率<5%) -> ✓ 通过


## 阶段七：工程师实战经验与进阶挑战 | Reflection & Engineering Insights

### 🛠️ 量产工程痛点与思考

1. **毫米波雷达加速度跳变与微分噪声 (Radar Doppler Differentiating)**：
   * 在实车传感器中，雷达直接测量的是径向速度（多普勒效应）。前车加速度 $a_{\text{lead}}$ 通常由速度数值差分得到，极易受到多径反射与杂波干扰导致“微分噪声爆炸”；
   * **工程落地解法**：量产 ADAS 控制器绝不直接使用原始差分值，而是必须通过卡尔曼滤波（Kalman Filter）或交互多模型算法（IMM）进行状态融合与平滑滤波。

2. **制动执行器液压建压延迟 (Hydraulic Pressure Build-up Lag)**：
   * 仿真中指令加速度可瞬间生效，但实车 ESC/EMB（电子机械制动）从接收 CAN 总线指令到卡钳建立 $80\,\%$ 制动压力，存在 $150 \sim 250\,\text{ms}$ 的物理滞后；
   * 在更高阶算法设计中，必须在 TTC 计算中将执行器响应延迟 $t_{\text{lag}}$ 提前计入制动安全边界。

3. **后车追尾风险与乘员舒适度 (Secondary Rear-end Collision)**：
   * 一味追求极早极强刹停，不仅降低了驾驶员舒适度，更极易引发后方紧跟车辆追尾；
   * 三级分级制动正是兼顾预充增压（减少制动空行程）、警示驾驶员与最后关头全力避险的行业标准做法。
